In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)

goldTablName = f"saleslake_{env}.gold_{env}.refinedcustomer"
# print(goldTablName)
silverTablName = f"saleslake_{env}.silver_{env}.cleanedCustomer"
# print(silverTablName)


In [0]:
spark.sql(f"""MERGE INTO {goldTablName} tgt
    USING
(   
WITH latest_cust_silver AS
    (
    SELECT * FROM {silverTablName}
        WHERE ingest_ts > (
                    SELECT coalesce(MAX(start_effective_ts),to_timestamp('1990-01-01','yyyy-MM-dd')) 
                    FROM {goldTablName}
                    )
    ),
    latest_rm_dup_silver AS
    (
    SELECT * FROM (
                    SELECT *, row_number() over (partition by customer_id order by ingest_ts desc) as rn 
                    FROM latest_cust_silver
                    ) WHERE rn = 1
    ),
    silver_gold_rec AS
    (
    SELECT  s.*,
    g.customer_sk AS customer_sk,
    g.is_active As is_active,
    g.rec_version AS rec_version , 
    g.start_effective_ts AS start_effective_ts,
    g.end_effective_ts AS  end_effective_ts,
    CASE WHEN g.customer_sk IS NULL THEN 1 ELSE g.rec_version + 1 END AS new_rec_version ,
    CASE WHEN g.customer_sk IS NULL THEN 'NEW'
         WHEN s.email <> g.email or s.address <> g.address or s.phone <> g.phone
         or s.city <> g.city or s.state <> g.state or s.country <> g.country or s.zip_code <> g.zip_code
         or s.segment <> g.segment
         THEN 'CHANGE'
         ELSE 'NO_CHANGE' END AS rec_flag
     FROM latest_rm_dup_silver s 
     LEFT JOIN ( SELECT * FROM {goldTablName} WHERE is_active = 'Y' ) g ON s.customer_id = g.customer_id   
    ),
    insert_flag AS
    (
        SELECT 
        NULL AS cust_merge_key,
        customer_id,	customer_name,	email,	phone,	address,	city,	state,	country,	zip_code,	segment,	    ingest_ts,	    customer_sk,	is_active,	rec_version,	start_effective_ts,	end_effective_ts,	new_rec_version,
        rec_flag,
        'INSERT' AS merge_flag
        FROM 	silver_gold_rec WHERE rec_flag in ('NEW','CHANGE')
    ),
        changed_flag AS
    (
        SELECT 
        customer_id AS cust_merge_key,
        customer_id,	customer_name,	email,	phone,	address,	city,	state,	country,	zip_code,	segment,	    ingest_ts,	    customer_sk,	is_active,	rec_version,	start_effective_ts,	end_effective_ts,	new_rec_version,
        rec_flag,  'UPDATE' AS merge_flag
        FROM 	silver_gold_rec WHERE rec_flag in ('CHANGE')
    )

    SELECT * FROM changed_flag
    UNION ALL
    SELECT * FROM insert_flag
  ) src
    ON tgt.customer_id = src.cust_merge_key
    WHEN MATCHED AND src.merge_flag = 'UPDATE' THEN
    UPDATE SET is_active ='N',
    end_effective_ts = current_timestamp()

WHEN NOT MATCHED AND src.merge_flag = 'INSERT' THEN INSERT 
    (
        customer_id,	customer_name,	email,	phone,	address,	city,	state,	country,	zip_code,	segment,is_active,	rec_version,	start_effective_ts,	end_effective_ts
    )
    VALUES 
    (
         customer_id,	customer_name,	email,	phone,	address,	city,	state,	country,	zip_code,	segment,'Y',new_rec_version	,	current_timestamp(),	to_timestamp('9999-12-31','yyyy-MM-dd')
    )

    """)

In [0]:
%sql
select * from saleslake_dev.gold_dev.refinedcustomer;